In [1]:
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
import awkward as ak
import uproot
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import boost_histogram as bh
import mplhep as hep
from cycler import cycler

mpl.rcParams['axes.prop_cycle'] = cycler(color=['blue', 'darkorange', 'green', 'k',  'red',  'violet', 'darkorange', 'cyan', 'gray'])
plt.style.use(hep.style.CMS) # use CMS style for plotting

# Dataset

In [2]:
import os

filepath = '/home/pku/fudawei/production/samples/HGamma'
gen_masses = [dir for dir in os.listdir(filepath) if dir.endswith('GeV')] + ['HGamma']
samples = {gen_mass: [] for gen_mass in gen_masses}
for (current_path, dirs, files) in os.walk(top='/home/pku/fudawei/production/samples/HGamma', topdown=True):
    ## topdown=False means depth-first
    postfix = current_path.split('/')[-1]
    if postfix in gen_masses:
        samples[postfix] = os.path.join(current_path, 'total.root')
    
samples

{'1000GeV': '/home/pku/fudawei/production/samples/HGamma/1000GeV/total.root',
 '1600GeV': '/home/pku/fudawei/production/samples/HGamma/1600GeV/total.root',
 '2200GeV': '/home/pku/fudawei/production/samples/HGamma/2200GeV/total.root',
 '2600GeV': '/home/pku/fudawei/production/samples/HGamma/2600GeV/total.root',
 '600GeV': '/home/pku/fudawei/production/samples/HGamma/600GeV/total.root',
 '1200GeV': '/home/pku/fudawei/production/samples/HGamma/1200GeV/total.root',
 '800GeV': '/home/pku/fudawei/production/samples/HGamma/800GeV/total.root',
 '900GeV': '/home/pku/fudawei/production/samples/HGamma/900GeV/total.root',
 '3500GeV': '/home/pku/fudawei/production/samples/HGamma/3500GeV/total.root',
 '1400GeV': '/home/pku/fudawei/production/samples/HGamma/1400GeV/total.root',
 '3000GeV': '/home/pku/fudawei/production/samples/HGamma/3000GeV/total.root',
 '700GeV': '/home/pku/fudawei/production/samples/HGamma/700GeV/total.root',
 '1800GeV': '/home/pku/fudawei/production/samples/HGamma/1800GeV/total.r

# From ROOT files to events

In [3]:
events = {
    m: NanoEventsFactory.from_root(file=samples[m], schemaclass=NanoAODSchema).events() for m in samples
}

### the benefit of using nanoevents rather than awkward array

In [4]:
set(dir(events['HGamma'].GenPart)) - set(dir(ak.Array(events['HGamma']).GenPart))

{'E',
 'FLAGS',
 'absolute',
 'add',
 'boost',
 'boostvec',
 'children',
 'cross',
 'delta_phi',
 'delta_r',
 'delta_r2',
 'distinctChildren',
 'distinctParent',
 'divide',
 'dot',
 'energy',
 'hasFlags',
 'mass2',
 'metric_table',
 'multiply',
 'nearest',
 'negative',
 'p',
 'p2',
 'parent',
 'pt2',
 'pvec',
 'px',
 'py',
 'pz',
 'r',
 'r2',
 'rho',
 'rho2',
 'subtract',
 'sum',
 't',
 'theta',
 'unit',
 'x',
 'y',
 'z'}

In [5]:
for gen_mass in events:
    print(gen_mass, 'events num:', len(events[gen_mass]))

1000GeV events num: 50000
1600GeV events num: 50000
2200GeV events num: 50000
2600GeV events num: 50000
600GeV events num: 50000
1200GeV events num: 50000
800GeV events num: 50000
900GeV events num: 50000
3500GeV events num: 50000
1400GeV events num: 50000
3000GeV events num: 50000
700GeV events num: 50000
1800GeV events num: 50000
2000GeV events num: 50000
2400GeV events num: 50000
HGamma events num: 750000


In [17]:
_events = events['HGamma']
cutflow = [len(_events)]

# Pre-selection
## Muon
muons = _events.Muon # (event, muon)
muon_cut = ( # (event, boolean)
    # high-pT cut-based ID (1 = tracker high pT, 2 = global high pT, which includes tracker high pT)
    (muons.highPtId == 2) & 
    (muons.tkRelIso < 0.1) & # Tracker-based relative isolation dR=0.3 for highPt, trkIso/tunePpt
    (abs(muons.eta) < 2.4) & 
    (muons.pt > 20) # I don't use `muon_corrected_pt` coming from ROOT.RoccoR
)
_events = _events[ak.sum(muon_cut, axis=1)==0] # No muon in pre-selected events
cutflow.append(len(_events))

## Electron
electrons = _events.Electron # (event, electron)
electron_cut = ( # (event, boolean)
    (electrons.cutBased_HEEP == True) & # cut-based HEEP ID
    (abs(electrons.eta) < 2.5) &
    (electrons.pt > 20)
)

_events = _events[ak.sum(electron_cut, axis=1)==0] # No electron in pre-selected events
cutflow.append(len(_events))

## Photon
photons = _events.Photon # (event, photon)
photon_cut = ( # (event, boolean)
    (photons.mvaID_WP90 > 0.2) &
    (photons.pt > 200) &
    (abs(photons.eta) < 2.4)
)
_events = _events[ak.sum(photon_cut, axis=1)==1] # Exactly 1 photon in pre-selected events
cutflow.append(len(_events))

## AK8 jet
AK8jets = _events.FatJet # (event, fatjet)
AK8jet_cut = ( # (event, boolean)
    (AK8jets.msoftdrop > 30) & # Corrected soft drop mass with PUPPI
    (AK8jets.pt > 250) & 
    (abs(AK8jets.eta) < 2.6) & 
    (AK8jets.jetId&2 > 0)
    # Jet ID flags bit1 is loose (always false in 2017 since it does not exist), bit2 is tight, bit3 is tightLepVeto
)
_events = _events[ak.sum(AK8jet_cut, axis=1)==1] # Exactly 1 AK8 jet in pre-selected events
cutflow.append(len(_events))

# Drop unselected photon and AK8 jet in events (Have to do again cuz FatJet cut not inherited)
photons = _events.Photon # (event, photon)
photon_cut = ( # (event, boolean)
    (photons.mvaID_WP90 > 0.2) &
    (photons.pt > 200) &
    (abs(photons.eta) < 2.4)
)
_events.Photon = _events.Photon[photon_cut]

AK8jets = _events.FatJet # (event, fatjet)
AK8jet_cut = ( # (event, boolean)
    (AK8jets.msoftdrop > 30) & # Corrected soft drop mass with PUPPI
    (AK8jets.pt > 250) & 
    (abs(AK8jets.eta) < 2.6) & 
    (AK8jets.jetId&2 > 0)
    # Jet ID flags bit1 is loose (always false in 2017 since it does not exist), bit2 is tight, bit3 is tightLepVeto
)
_events.FatJet = _events.FatJet[AK8jet_cut]

## Photon-Jet cleaning
jet_cleaning = (_events.FatJet.delta_r(_events.Photon) > 0.8)
_events = _events[ak.sum(jet_cleaning, axis=1)>0]
cutflow.append(len(_events))

# Construct NanoAnalysis processor with pre-selection and gen-matching

In [39]:
import awkward as ak
from coffea import processor
from coffea.nanoevents.methods import candidate

class HGammaProcessor(processor.ProcessorABC):
    def __init__(self):
        pass
    
    def __preSelect(self, _events):
        cutflow = [len(_events)]

        # Pre-selection
        ## Muon
        muons = _events.Muon # (event, muon)
        muon_cut = ( # (event, boolean)
            # high-pT cut-based ID (1 = tracker high pT, 2 = global high pT, which includes tracker high pT)
            (muons.highPtId == 2) & 
            (muons.tkRelIso < 0.1) & # Tracker-based relative isolation dR=0.3 for highPt, trkIso/tunePpt
            (abs(muons.eta) < 2.4) & 
            (muons.pt > 20) # I don't use `muon_corrected_pt` coming from ROOT.RoccoR
        )
        _events = _events[ak.sum(muon_cut, axis=1)==0] # No muon in pre-selected events
        cutflow.append(len(_events))

        ## Electron
        electrons = _events.Electron # (event, electron)
        electron_cut = ( # (event, boolean)
            (electrons.cutBased_HEEP == True) & # cut-based HEEP ID
            (abs(electrons.eta) < 2.5) &
            (electrons.pt > 20)
        )

        _events = _events[ak.sum(electron_cut, axis=1)==0] # No electron in pre-selected events
        cutflow.append(len(_events))

        ## Photon
        photons = _events.Photon # (event, photon)
        photon_cut = ( # (event, boolean)
            (photons.mvaID_WP90 > 0.2) &
            (photons.pt > 200) &
            (abs(photons.eta) < 2.4)
        )
        _events = _events[ak.sum(photon_cut, axis=1)==1] # Exactly 1 photon in pre-selected events
        cutflow.append(len(_events))

        ## AK8 jet
        AK8jets = _events.FatJet # (event, fatjet)
        AK8jet_cut = ( # (event, boolean)
            (AK8jets.msoftdrop > 30) & # Corrected soft drop mass with PUPPI
            (AK8jets.pt > 250) & 
            (abs(AK8jets.eta) < 2.6) & 
            (AK8jets.jetId&2 > 0)
            # Jet ID flags bit1 is loose (always false in 2017 since it does not exist), bit2 is tight, bit3 is tightLepVeto
        )
        _events = _events[ak.sum(AK8jet_cut, axis=1)==1] # Exactly 1 AK8 jet in pre-selected events
        cutflow.append(len(_events))
        
        ## Photon-Jet cleaning
        jet_cleaning = (_events.FatJet.delta_r(_events.Photon) > 0.8)
        _events = _events[jet_cleaning]
        cutflow.append(len(_events))
        
        return _events, cutflow
    
    def __genMatching(self, events):
        "statusFlags().isLastCopyBeforeFSR()                  * 16384 +"
        "statusFlags().isLastCopy()                           * 8192  +"
        "statusFlags().isFirstCopy()                          * 4096  +"
        "statusFlags().fromHardProcessBeforeFSR()             * 2048  +"
        "statusFlags().isDirectHardProcessTauDecayProduct()   * 1024  +"
        "statusFlags().isHardProcessTauDecayProduct()         * 512   +"
        "statusFlags().fromHardProcess()                      * 256   +"
        "statusFlags().isHardProcess()                        * 128   +"
        "statusFlags().isDirectHadronDecayProduct()           * 64    +"
        "statusFlags().isDirectPromptTauDecayProduct()        * 32    +"
        "statusFlags().isDirectTauDecayProduct()              * 16    +"
        "statusFlags().isPromptTauDecayProduct()              * 8     +"
        "statusFlags().isTauDecayProduct()                    * 4     +"
        "statusFlags().isDecayedLeptonHadron()                * 2     +"
        "statusFlags().isPrompt()                             * 1      "
        gen_Higgs = events.GenPart[ # (event, particle)
            (events.GenPart.pdgId == 25) & # select all Higgs
            (events.GenPart.hasFlags(['isLastCopy'])) # Higgs is the last copy
        ] # (event, Higgs)

        gen_gamma = events.GenPart[ # (event, particle)
            (events.GenPart.pdgId == 22) & # select all gamma
            (events.GenPart.hasFlags(['isLastCopy'])) # gamma is the last copy
        ] # (event, gamma)
        
        gen_H_children = gen_Higgs.children
        gen_HWW = gen_H_children[ # (event, Higgs, W)
            (ak.num(gen_H_children, axis=2) == 2) & # Higgs has two children
            ak.all(np.abs(gen_H_children.pdgId) == 24, axis=2) # both childen are W 
        ] # (event, Higgs, W)

        gen_HWW_children = gen_HWW.children
        gen_HWW_children = gen_HWW_children[
            ak.all(ak.num(gen_HWW_children, axis=3) == 2, axis=2) # both W has two children
        ] # (event, Higgs, W, W_child)

        gen_HWW_Wtag = ( # (event, Higgs, W)
            ak.zeros_like(ak.any(gen_HWW_children.pdgId, axis=3)) +
            1 * ak.any(abs(gen_HWW_children.pdgId)==11, axis=3) + # one W decays to e
            2 * ak.any(abs(gen_HWW_children.pdgId)==13, axis=3) + # one W decays to mu
            4 * ak.any(abs(gen_HWW_children.pdgId)==15, axis=3) + # one W decays to tau
            8 * ak.all(abs(gen_HWW_children.pdgId)<7, axis=3)   # one W decays to qq
        ) # (event, Higgs, W)
        gen_HWW_Wtag = gen_HWW_Wtag[ak.sum(gen_HWW_Wtag, axis=2, keepdims=False)>8] # at least one W-->qq
        gen_HWW_tag = ak.sum(gen_HWW_Wtag, axis=2) # (event, Higgs)
        
        cut = (
            (ak.num(gen_gamma, axis=1)>0) &
            (ak.num(gen_HWW_tag, axis=1)>0)
        )
        
        gen_HGamma_events = events[cut] # (event, )
        
        return gen_HGamma_events, gen_HWW_tag
    
    def process(self, events):
        preselected_events, cutflow = self.__preSelect(events)
        gen_HGamma_events, gen_HWW_tag = self.__genMatching(events)
        return preselected_events, cutflow, gen_HGamma_events, gen_HWW_tag

    def postprocess(self, accumulator):
        pass

# Start running

In [40]:
p = HGammaProcessor()
preselected_events = {m: None for m in events}
cutflow = {m: None for m in events}
gen_HGamma_events = {m: None for m in events}
gen_HWW_tag = {m: None for m in events}

for m in events:
    preselected_events[m], cutflow[m], gen_HGamma_events[m], gen_HWW_tag = p.process(events[m])
preselected_events, gen_HGamma_events

({'1000GeV': None,
  '1600GeV': None,
  '2200GeV': None,
  '2600GeV': None,
  '600GeV': None,
  '1200GeV': None,
  '800GeV': None,
  '900GeV': None,
  '3500GeV': None,
  '1400GeV': None,
  '3000GeV': None,
  '700GeV': None,
  '1800GeV': None,
  '2000GeV': None,
  '2400GeV': None,
  'HGamma': <NanoEventsArray [<event 1:1:6>, ... <event 1:48:47995>] type='376626 * event'>},
 {'1000GeV': None,
  '1600GeV': None,
  '2200GeV': None,
  '2600GeV': None,
  '600GeV': None,
  '1200GeV': None,
  '800GeV': None,
  '900GeV': None,
  '3500GeV': None,
  '1400GeV': None,
  '3000GeV': None,
  '700GeV': None,
  '1800GeV': None,
  '2000GeV': None,
  '2400GeV': None,
  'HGamma': <NanoEventsArray [<event 1:1:6>, ... <event 1:48:48000>] type='128884 * event'>})

# Store output
### It takes long time, be patient

In [8]:
ak.to_parquet(array=preselected_events['HGamma'], where='./output/preselected_events.parq')
ak.to_parquet(array=gen_HGamma_events['HGamma'], where='./output/gen_HGamma_events.parq')

/home/olympus/fudawei/anaconda3/lib/python3.9/site-packages/awkward/operations/convert.py:1999: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if distutils.version.LooseVersion(
/home/olympus/fudawei/anaconda3/lib/python3.9/site-packages/awkward/operations/convert.py:2001: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  ) < distutils.version.LooseVersion("2.0.0"):
/home/olympus/fudawei/anaconda3/lib/python3.9/site-packages/awkward/_connect/_numba/__init__.py:30: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if not checked_version and distutils.version.LooseVersion(
/home/olympus/fudawei/anaconda3/lib/python3.9/site-packages/awkward/_connect/_numba/__init__.py:32: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  ) < distutils.version.LooseVersion("0.50"):
/home/olympus/fudawei/anaconda3/lib/python3.9

In [ ]:
def plot(labels, filename, distributions, bins=10, x_min=0, x_max=10, density=False, stack=False, colorlist=['red', 'blue', 'green', 'black', 'cyan', 'darkorange', 'darkviolet', 'SlateGray', 'HotPink', 'LightSkyBlue']):
    ## canvas initializing
    mpl.rcParams['axes.prop_cycle'] = cycler(color=colorlist)
    #f, ax = plt.subplots()
    plt.figure(figsize=(8,8))
    ax=plt.gca()
    plt.grid()
    hep.cms.label(data=False, year=2018, ax=ax)
    
    
    ## plot
    for i in range(len(labels)):
        hist = bh.Histogram(bh.axis.Regular(bins, x_min, x_max), storage=bh.storage.Weight())
        var = distributions[i]
        hist.fill(var)
        h, err = hist.view().value, np.sqrt(hist.view().variance)
        hep.histplot(h, bins=hist.axes[0].edges, yerr=err, label=labels[i], histtype='step', density=density, stack=stack)


    ## axises
    plt.xlim(x_min, x_max)
    plt.ylim(0, ax.get_ylim()[1]*1.1)
    ax.ticklabel_format(useOffset=False, style='plain')
    #x_major_locator=plt.MultipleLocator(bin_width*8 if variable=='fj_gen_mass' else bin_width*2)
    #y_major_locator=MultipleLocator(1)
    #ax.xaxis.set_major_locator(x_major_locator)
    #ax.yaxis.set_major_locator(y_major_locator)
    if density==False:
        plt.ylabel('Events', fontsize=20, ha='right', y=1)
    elif density==True:    
        plt.ylabel('A.U.', fontsize=20, ha='right', y=1)
    plt.xlabel(r'$N_{AK8}$', fontsize=20, ha='right', x=1)

    plt.xticks(size=14)
    plt.yticks(size=14)


    ## title, text and legend
    #plt.title('ROC Curve of HWW4q vs. QCD', fontsize=24,color="black")
    plt.legend(loc="best", ncol=1, frameon=False,fontsize=18)
    #plt.text(0.96*ax.get_xlim()[1], ax.get_ylim()[1]*0.72,"At "+r"$\mathrm{m^{gen}_{H}=}$"+f"{Higgsmass} GeV", fontsize=24, color="black", ha='right')


    ## finalizing
    suffix=''
    if density==True:   suffix+='_AU'
    if stack==True:     suffix+='_stack'
    plt.savefig("plots/"+filename+suffix+".pdf", bbox_inches='tight')
    plt.show()